Data Preprocessing

In [1]:
import pandas as pd

url = 'http://archive.ics.uci.edu/ml/' \
'machine-learning-databases/auto-mpg/auto-mpg.data'
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower',
    'Weight', 'Acceleration', 'Model Year', 'Origin']

df = pd.read_csv(url, names=column_names, na_values='?', comment='\t',
                sep=' ', skipinitialspace=True)
df = df.dropna()
df.reset_index(drop=True)


,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
0,18.0,8,307.0,130.0,3504.0,12.0,70,1
1,15.0,8,350.0,165.0,3693.0,11.5,70,1
2,18.0,8,318.0,150.0,3436.0,11.0,70,1
3,16.0,8,304.0,150.0,3433.0,12.0,70,1
4,17.0,8,302.0,140.0,3449.0,10.5,70,1
...,...,...,...,...,...,...,...,...
387,27.0,4,140.0,86.0,2790.0,15.6,82,1
388,44.0,4,97.0,52.0,2130.0,24.6,82,2
389,32.0,4,135.0,84.0,2295.0,11.6,82,1
390,28.0,4,120.0,79.0,2625.0,18.6,82,1


In [2]:
import sklearn
import sklearn.model_selection

df_train, df_test = sklearn.model_selection.train_test_split(
    df, train_size=0.8, random_state=1
)
train_stats = df_train.describe().transpose()

numeric_column_names = [
'Cylinders', 'Displacement',
'Horsepower', 'Weight',
'Acceleration'
]

In [3]:
df_train_norm, df_test_norm = df_train.copy(), df_test.copy()

for col in numeric_column_names:
    mean, std = train_stats.loc[col, 'mean'], train_stats.loc[col, 'std']
    df_train_norm[col] = (df_train_norm[col] - mean) / std
    df_test_norm[col] = (df_test_norm[col] - mean) / std
df_train_norm.tail()

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
205,28.0,-0.824303,-0.901020,-0.736562,-0.950031,0.255202,76,3
257,19.4,0.351127,0.413800,-0.340982,0.293190,0.548737,78,1
73,13.0,1.526556,1.144256,0.713897,1.339617,-0.625403,72,1
237,30.5,-0.824303,-0.891280,-1.053025,-1.072585,0.475353,77,1
38,14.0,1.526556,1.563051,1.636916,1.470420,-1.359240,71,1


In [4]:
# turn modelYear into numerical ordinal feature(bucket them into 4 categories)
import torch

boundaries = torch.tensor([73, 76, 79])
v = torch.tensor(df_train_norm['Model Year'].values)
df_train_norm['Model Year Bucketed'] = torch.bucketize(
    v, boundaries=boundaries, right=True)

v = torch.tensor(df_test_norm['Model Year'].values)
df_test_norm['Model Year Bucketed'] = torch.bucketize(
    v, boundaries, right=True
)
numeric_column_names.append('Model Year Bucketed')

In [5]:
# do one hot encoding for Origin column
from torch.nn.functional import one_hot
total_origin = len(set(df_train_norm['Origin']))

origin_encoded = one_hot(
    torch.tensor(torch.from_numpy(df_train_norm['Origin'].values)%total_origin)
)

X_train_numeric = torch.Tensor(df_train_norm[numeric_column_names].values)
X_train = torch.cat([X_train_numeric, origin_encoded], 1).float()

origin_encoded = one_hot(
    torch.tensor(torch.from_numpy(df_test_norm['Origin'].values)%total_origin)
)
X_test_numeric = torch.Tensor(df_test_norm[numeric_column_names].values)
X_test = torch.cat([X_test_numeric, origin_encoded], 1).float()

y_train = torch.tensor(df_train_norm['MPG'].values).float()
y_test = torch.tensor(df_test_norm['MPG'].values).float()


C:\Users\Andrew\AppData\Local\Temp\ipykernel_15528\2119212857.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.tensor(torch.from_numpy(df_train_norm['Origin'].values)%total_origin)
C:\Users\Andrew\AppData\Local\Temp\ipykernel_15528\2119212857.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(torch.from_numpy(df_train_norm['Origin'].values)%total_origin)
C:\Users\Andrew\AppData\Local\Temp\ipykernel_15528\211921285

Train the DNN

In [6]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train, y_train)
batch_size = 8
torch.manual_seed(1)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

In [7]:
import torch.nn as nn

hidden_units = [8, 4]
input_size = X_train.shape[1]
all_layers = []

for hidden_unit in hidden_units:
    layer = nn.Linear(input_size, hidden_unit)
    all_layers.append(layer)
    all_layers.append(nn.ReLU())
    input_size = hidden_unit

all_layers.append(nn.Linear(input_size, 1))

model = nn.Sequential(*all_layers)
model


Sequential(
  (0): Linear(in_features=9, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)

In [8]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [9]:
num_epochs = 201
log_epochs = 20
torch.manual_seed(1)

for epoch in range(num_epochs):
    train_loss = 0.0
    for xb, yb in train_dl:
        preds = model(xb)[:,0]
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_loss += loss.item()
    
    if epoch % log_epochs == 0:
        print(f"Epoch {epoch} - Train Loss: {train_loss/len(train_dl)}")

Epoch 0 - Train Loss: 536.1046592712403
Epoch 20 - Train Loss: 8.436100562382489
Epoch 40 - Train Loss: 7.869508296251297
Epoch 60 - Train Loss: 7.189111997187138
Epoch 80 - Train Loss: 6.706427782401443
Epoch 100 - Train Loss: 6.760309821367263
Epoch 120 - Train Loss: 6.310717681050301
Epoch 140 - Train Loss: 6.688404183089733
Epoch 160 - Train Loss: 6.75493400990963
Epoch 180 - Train Loss: 6.202939441800117
Epoch 200 - Train Loss: 6.119392393529415


In [10]:
with torch.no_grad():
    pred = model(X_test)[:, 0]
    loss = loss_fn(pred, y_test)
    print(f'Test MSE: {loss.item():.4f}')
    print(f'Test MAE: {nn.L1Loss()(pred, y_test).item():.4f}')

Test MSE: 9.0524
Test MAE: 1.9479
